# Flash-MaxSim: Interactive Demo

**Fused GPU kernel for ColBERT/ColPali MaxSim scoring — up to 13x faster, 143x less memory.**

This notebook demonstrates Flash-MaxSim on a real ColBERT model with real text queries, including:
1. Encoding real queries & passages with a pretrained model
2. Scoring: Naive PyTorch vs Flash-MaxSim (correctness + speed)
3. INT8 quantized scoring (2x compression, same accuracy)
4. Speedup plots across different scales
5. Memory comparison

In [ ]:
# Setup
import time
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

assert torch.cuda.is_available(), "This demo requires a CUDA GPU"
torch.zeros(1, device='cuda')  # init CUDA context
print(f"GPU: {torch.cuda.get_device_name()}")

from flash_maxsim import flash_maxsim, flash_maxsim_batched, flash_maxsim_train, maxsim_naive
from flash_maxsim import flash_maxsim_int8, quantize_int8

def bench(fn, *a, warmup=5, n=50):
    for _ in range(warmup): fn(*a)
    torch.cuda.synchronize()
    t = []
    for _ in range(n):
        torch.cuda.synchronize(); s = time.perf_counter(); fn(*a)
        torch.cuda.synchronize(); t.append((time.perf_counter()-s)*1000)
    t.sort(); return t[len(t)//2]

## 1. Load a Real ColBERT Model & Encode Text

In [ ]:
# Load model (falls back to simulated if PyLate not available)
queries = [
    "What is information retrieval?",
    "How does ColBERT work?",
    "What are transformer models used for?",
    "Explain neural search engines",
    "What is late interaction in retrieval?",
]
passages = [
    "Information retrieval is the process of finding relevant documents from a large collection based on a user query.",
    "ColBERT uses late interaction between query and document token embeddings to score relevance efficiently.",
    "Transformer models are deep learning architectures used for NLP tasks like translation, summarization, and search.",
    "Neural search engines use learned representations to match queries with documents based on semantic meaning.",
    "Late interaction computes token-level similarities between queries and documents, then aggregates via MaxSim.",
    "Traditional search engines rely on keyword matching using inverted indices like BM25.",
    "Dense retrieval encodes queries and documents as single vectors and uses approximate nearest neighbor search.",
    "ColPali extends ColBERT to visual document retrieval by treating document pages as image patches.",
    "Flash Attention avoids materializing the attention matrix by tiling computation in SRAM.",
    "Quantization reduces model size by representing weights with fewer bits, such as INT8 or INT4.",
]

try:
    print("Loading ColBERT model (first import may take a minute)...", flush=True)
    from pylate import models
    model = models.ColBERT("answerdotai/answerai-colbert-small-v1")
    print(f"Loaded: answerai-colbert-small-v1")
    q_embs = model.encode(queries, is_query=True)
    p_embs = model.encode(passages)
    max_lq = max(e.shape[0] for e in q_embs)
    max_lp = max(e.shape[0] for e in p_embs)
    d = q_embs[0].shape[1]
    Q = torch.zeros(len(queries), max_lq, d, device='cuda', dtype=torch.float16)
    for i, e in enumerate(q_embs):
        t = torch.tensor(np.array(e), device='cuda', dtype=torch.float16)
        Q[i, :t.shape[0]] = t
    D = torch.zeros(len(passages), max_lp, d, device='cuda', dtype=torch.float16)
    for i, e in enumerate(p_embs):
        t = torch.tensor(np.array(e), device='cuda', dtype=torch.float16)
        D[i, :t.shape[0]] = t
    REAL_MODEL = True
except ImportError:
    print("PyLate not installed — using simulated embeddings")
    print("Install with: pip install pylate sentence-transformers")
    d = 128
    Q = F.normalize(torch.randn(len(queries), 32, d, device='cuda', dtype=torch.float16), dim=-1)
    D = F.normalize(torch.randn(len(passages), 300, d, device='cuda', dtype=torch.float16), dim=-1)
    REAL_MODEL = False

print(f"Q: {Q.shape}, D: {D.shape}")

## 2. Scoring: Naive vs Flash-MaxSim

Score all queries against all passages. Compare results and rankings.

In [ ]:
# Score with both methods
scores_naive = torch.einsum('nqd,pld->npql', Q.float(), D.float()).max(3).values.sum(2)
scores_flash = flash_maxsim_batched(Q, D, shared_docs=True)

# Compare rankings
print("Query → Best Passage (Naive vs Flash)\n")
for i, q in enumerate(queries):
    n_rank = scores_naive[i].argsort(descending=True)[:3].tolist()
    f_rank = scores_flash[i].argsort(descending=True)[:3].tolist()
    match = "✅ MATCH" if n_rank == f_rank else "❌ DIFFER"
    print(f'  Q: "{q}"')
    print(f'  → "{passages[f_rank[0]]}"')
    print(f'  Naive: {n_rank}  Flash: {f_rank}  {match}\n')

## 3. Speedup Across Scales

Benchmark Flash-MaxSim vs naive across increasing problem sizes and plot the results.

In [ ]:
# Sweep: varying B
configs_B = [
    (100, 32, 300, 128, "B=100"),
    (500, 32, 300, 128, "B=500"),
    (1000, 32, 300, 128, "B=1000"),
    (2000, 32, 300, 128, "B=2000"),
]

# Sweep: varying Lq with Ld=1024 (ColBERT → ColPali regime)
configs_Lq = [
    (500, 32, 1024, 128, "Lq=32"),
    (500, 64, 1024, 128, "Lq=64"),
    (500, 128, 1024, 128, "Lq=128"),
]

# Sweep: Lq=1024 regime (the big numbers)
configs_1024 = [
    (100, 1024, 1024, 128, "B=100"),
    (500, 1024, 1024, 128, "B=500"),
    (1000, 1024, 1024, 128, "B=1000"),
    (5000, 1024, 1024, 128, "B=5000"),
]

def naive_fp32(Q, D):
    return torch.einsum('qd,bld->bql', Q.float(), D.float()).max(2).values.sum(1)

def run_sweep(configs):
    labels, naive_ms, flash_ms, speedups = [], [], [], []
    for B, Lq, Ld, d, label in configs:
        Qs = F.normalize(torch.randn(Lq, d, device='cuda', dtype=torch.float16), dim=-1)
        Ds = F.normalize(torch.randn(B, Ld, d, device='cuda', dtype=torch.float16), dim=-1)
        n = bench(naive_fp32, Qs, Ds, warmup=5, n=30)
        f = bench(flash_maxsim, Qs, Ds, warmup=5, n=30)
        labels.append(label); naive_ms.append(n); flash_ms.append(f); speedups.append(n/f)
        print(f"  {label}: naive={n:.2f}ms  flash={f:.2f}ms  {n/f:.1f}x")
    return labels, naive_ms, flash_ms, speedups

print("Sweep B (Lq=32, Ld=300):")
labels_B, naive_B, flash_B, sp_B = run_sweep(configs_B)

print("\nSweep Lq (B=500, Ld=1024 — ColBERT to ColPali):")
labels_Lq, naive_Lq, flash_Lq, sp_Lq = run_sweep(configs_Lq)

print("\nSweep B at Lq=Ld=1024 (ColPali image-to-image):")
labels_1024, naive_1024, flash_1024, sp_1024 = run_sweep(configs_1024)

In [ ]:
# Plot all three sweeps
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

def plot_bars(ax, labels, naive, flash, speedups, title):
    x = np.arange(len(labels)); w = 0.3
    ax.bar(x-w/2, naive, w, label='Naive FP32', color='#FF7043', edgecolor='black', lw=0.4)
    ax.bar(x+w/2, flash, w, label='Flash-MaxSim', color='#1E88E5', edgecolor='black', lw=0.4)
    for i, sp in enumerate(speedups):
        ax.text(i+w/2, flash[i]+max(naive)*0.03, f'{sp:.1f}x', ha='center', fontsize=10, fontweight='bold', color='#1E88E5')
    ax.set_ylabel('Latency (ms)'); ax.set_title(title)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.legend(fontsize=8)

plot_bars(axes[0], labels_B, naive_B, flash_B, sp_B, 'ColBERT: Vary B\n(Lq=32, Ld=300)')
plot_bars(axes[1], labels_Lq, naive_Lq, flash_Lq, sp_Lq, 'ColBERT→ColPali: Vary Lq\n(B=500, Ld=1024)')
plot_bars(axes[2], labels_1024, naive_1024, flash_1024, sp_1024, 'ColPali Image: Lq=Ld=1024\n(the big numbers!)')

plt.tight_layout(); plt.show()
print("ColPali regime (Lq=Ld=1024) shows 10-13x speedup!")

## 4. INT8 Quantized Scoring

2x compression with zero ranking loss. Dequantization happens in SRAM — no HBM penalty.

In [ ]:
# INT8 quantization and scoring
B, Lq, Ld, d_dim = 1000, 32, 300, 128
Qs = F.normalize(torch.randn(Lq, d_dim, device='cuda', dtype=torch.float16), dim=-1)
Ds = F.normalize(torch.randn(B, Ld, d_dim, device='cuda', dtype=torch.float16), dim=-1)

D_q, scales, mins = quantize_int8(Ds)
print(f"Storage: FP16={Ds.nbytes/1e6:.1f}MB  INT8={D_q.nbytes/1e6:.1f}MB  ({Ds.nbytes/D_q.nbytes:.0f}x compression)")

# Speed comparison
def naive_int8():
    Df = D_q.float()*scales.float()+mins.float()
    return torch.einsum('qd,bld->bql',Qs.float(),Df).max(2).values.sum(1)

nf = bench(naive_fp32, Qs, Ds, warmup=5, n=30)
ni = bench(naive_int8, warmup=5, n=30)
fi = bench(flash_maxsim_int8, Qs, D_q, scales, mins, warmup=5, n=30)

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
methods = ['Naive FP32', 'Naive INT8\n(dequant penalty!)', 'Flash FP16', 'Flash Q8\n(fused dequant)']
times = [nf, ni, bench(flash_maxsim, Qs, Ds, warmup=5, n=30), fi]
colors = ['#9E9E9E', '#CE93D8', '#1E88E5', '#2E7D32']
bars = ax.bar(methods, times, color=colors, edgecolor='black', lw=0.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x()+bar.get_width()/2, t+max(times)*0.02, f'{t:.3f}ms', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Latency (ms)')
ax.set_title(f'Quantization: Naive INT8 is SLOWER, Flash Q8 is same speed\n(B={B}, Lq={Lq}, Ld={Ld}, d={d_dim})')
plt.tight_layout(); plt.show()

print(f"\nNaive INT8 is {ni/nf:.1f}x SLOWER than FP32 (dequant penalty)")
print(f"Flash Q8 is {nf/fi:.1f}x FASTER than FP32 (penalty eliminated)")

## 5. Peak Memory Comparison

Flash-MaxSim never allocates the similarity matrix — it's computed in SRAM tiles.

In [ ]:
# Memory comparison across scales
mem_configs = [
    (1, 500, 32, 1024, 128, "1q×500p\nLd=1024"),
    (1, 1000, 32, 1024, 128, "1q×1000p\nLd=1024"),
    (1, 1000, 1024, 1024, 128, "1q×1000p\nLq=Ld=1024"),
]

naive_mems, flash_mems, labels_mem = [], [], []
for Nq, B, Lq, Ld, d_dim, label in mem_configs:
    Qm = F.normalize(torch.randn(Nq, Lq, d_dim, device='cuda', dtype=torch.float16), dim=-1)
    Dm = F.normalize(torch.randn(B, Ld, d_dim, device='cuda', dtype=torch.float16), dim=-1)
    sim_gb = Nq*B*Lq*Ld*4/1e9
    
    # Naive
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    try:
        _ = torch.einsum('nqd,bld->nbql', Qm.float(), Dm.float()).max(3).values.sum(2)
        torch.cuda.synchronize()
        n_gb = (torch.cuda.max_memory_allocated()-base)/1e9
        del _
    except: n_gb = sim_gb
    torch.cuda.empty_cache()
    
    # Flash
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    _ = flash_maxsim_batched(Qm, Dm, shared_docs=True); torch.cuda.synchronize()
    f_gb = (torch.cuda.max_memory_allocated()-base)/1e9
    del _; torch.cuda.empty_cache()
    
    naive_mems.append(n_gb); flash_mems.append(f_gb); labels_mem.append(label)
    del Qm, Dm; torch.cuda.empty_cache()

# Plot
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(labels_mem)); w = 0.3
ax.bar(x-w/2, naive_mems, w, label='Naive einsum', color='#FF7043', edgecolor='black', lw=0.5)
ax.bar(x+w/2, flash_mems, w, label='Flash-MaxSim', color='#1E88E5', edgecolor='black', lw=0.5)
for i, (n, f) in enumerate(zip(naive_mems, flash_mems)):
    ax.text(i-w/2, n+0.1, f'{n:.1f}GB', ha='center', fontsize=9, fontweight='bold', color='#FF7043')
    ax.text(i+w/2, max(f,0.05)+0.1, f'{f:.1f}GB', ha='center', fontsize=9, fontweight='bold', color='#1E88E5')
    if f > 0.001:
        ax.text(i, max(n,f)*1.15, f'{n/max(f,0.001):.0f}x less', ha='center', fontsize=10, color='green', fontweight='bold')
ax.set_ylabel('Peak GPU Memory (GB)'); ax.set_title('Flash-MaxSim: Similarity matrix never in GPU memory')
ax.set_xticks(x); ax.set_xticklabels(labels_mem); ax.legend()
plt.tight_layout(); plt.show()

## 6. Try Your Own Queries!

Edit the queries and passages below and re-run to see Flash-MaxSim on your own data.

In [ ]:
# ✏️ Edit these and re-run!
my_queries = [
    "How to make coffee",
    "machine learning basics",
]

my_passages = [
    "To make coffee, grind fresh beans, add hot water, and filter through a paper filter.",
    "Machine learning is a subset of AI where models learn patterns from data.",
    "The weather tomorrow will be sunny with temperatures around 75 degrees.",
    "Python is a popular programming language for data science.",
    "Espresso is made by forcing hot water through finely ground coffee beans.",
]

if REAL_MODEL:
    q_e = model.encode(my_queries, is_query=True)
    p_e = model.encode(my_passages)
    import numpy as np
    mlq = max(e.shape[0] for e in q_e); mlp = max(e.shape[0] for e in p_e); dd = q_e[0].shape[1]
    Qm = torch.zeros(len(my_queries), mlq, dd, device='cuda', dtype=torch.float16)
    for i, e in enumerate(q_e): Qm[i,:e.shape[0]] = torch.tensor(np.array(e), device='cuda', dtype=torch.float16)
    Dm = torch.zeros(len(my_passages), mlp, dd, device='cuda', dtype=torch.float16)
    for i, e in enumerate(p_e): Dm[i,:e.shape[0]] = torch.tensor(np.array(e), device='cuda', dtype=torch.float16)
else:
    dd = 128
    Qm = F.normalize(torch.randn(len(my_queries), 32, dd, device='cuda', dtype=torch.float16), dim=-1)
    Dm = F.normalize(torch.randn(len(my_passages), 300, dd, device='cuda', dtype=torch.float16), dim=-1)

scores = flash_maxsim_batched(Qm, Dm, shared_docs=True)

print("Results:\n")
for i, q in enumerate(my_queries):
    ranked = scores[i].argsort(descending=True).tolist()
    print(f'  Query: "{q}"')
    for rank, idx in enumerate(ranked[:3]):
        print(f'    #{rank+1}: "{my_passages[idx]}" (score={scores[i,idx]:.2f})')
    print()